## NLLB: машинный перевод и перенос на другую задачу



Модель распространяется с ограничениями на использование, в том числе некоммерческими. Перед применением в собственном проекте изучите её карточку и лицензию.

In [ ]:
# %pip -q install "transformers>=4.45,<5" "datasets>=3,<5" "accelerate>=1,<2" "sacrebleu>=2" "sentencepiece>=0.2" "evaluate>=0.4,<1" "rouge_score>=0.1.2" "pandas>=2"

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed,
)

# ===== ПЕРЕМЕННЫЕ =====
MODEL_ID = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "rus_Cyrl"
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPERIMENT_TRAIN_SIZE = 64
EXPERIMENT_VALID_SIZE = 16
MAX_SOURCE_LENGTH = 192
MAX_TARGET_LENGTH = 64
OUTPUT_DIR = Path("./outputs_nllb_experiment")
set_seed(SEED)
print("Устройство:", DEVICE)

/home/dasha/work/NLP_ODS_autumn_2026/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cuda


#### Чем NLLB отличается от T5?

Обе модели используют архитектуру «кодировщик — декодировщик», но **целевое предобучение различно**. NLLB ориентирована на перевод между многими языками. Управление осуществляется в том числе специальными языковыми идентификаторами, а не свободно сформулированной инструкцией.

`src_lang` определяет язык входного текста. `forced_bos_token_id` задаёт токен целевого языка в начале декодирования. Например, `eng_Latn` — английский язык (латиница), `rus_Cyrl` — русский язык (кириллица).

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

def translate(text, source="eng_Latn", target="rus_Cyrl", beams=4):
    tokenizer.src_lang = source
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    target_token = tokenizer.convert_tokens_to_ids(target)
    if target_token == tokenizer.unk_token_id:
        raise ValueError(f"Неизвестный код языка: {target}")
    with torch.inference_mode():
        generated = model.generate(**encoded, forced_bos_token_id=target_token,
                                   max_new_tokens=100, num_beams=beams)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

examples = [
  "The company announced a new research center in Berlin.",
  "The annual revenue increased by 12 percent in 2025.",
  "The board did not approve the proposed agreement.",
]
for line in examples:
    print("Оригинал:", line)
    print("Перевод:", translate(line), "\n")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 7532.76it/s]


Оригинал: The company announced a new research center in Berlin.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Перевод: Компания объявила о создании нового исследовательского центра в Берлине. 

Оригинал: The annual revenue increased by 12 percent in 2025.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Перевод: Ежегодные доходы увеличились на 12 процентов в 2025 году. 

Оригинал: The board did not approve the proposed agreement.
Перевод: Совет директоров не одобрил предложенное соглашение. 



## 2. Управление языками и обратный перевод

Обратный перевод — самостоятельная генерация, а не математическая обратная функция. Совпадение исходной и восстановленной фразы не обязательно; несовпадение также не доказывает ошибку перевода.

In [3]:
source = "The central bank left the interest rate unchanged."
russian = translate(source, "eng_Latn", "rus_Cyrl")
back = translate(russian, "rus_Cyrl", "eng_Latn")
print("Исходный:", source)
print("На русском:", russian)
print("После обратного перевода:", back)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Исходный: The central bank left the interest rate unchanged.
На русском: Центральный банк оставил процентную ставку неизменной.
После обратного перевода: The central bank has kept the interest rate unchanged.


## 3. Как оценивать перевод?

BLEU оценивает совпадение последовательностей слов с эталонным переводом, chrF — совпадение символьных n-грамм. Обе метрики зависят от эталона и не гарантируют сохранение фактов. Ниже оценки на **двух искусственно заданных примерах**, их нельзя интерпретировать как качество на тестовом корпусе.

In [4]:
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
sources = ["The meeting starts on Monday.", "The company did not increase prices."]
references = ["Встреча начинается в понедельник.", "Компания не повышала цены."]
predictions = [translate(x) for x in sources]
print(pd.DataFrame({"Исходный текст": sources, "Эталон": references, "Перевод": predictions}).to_string(index=False))
print("BLEU:", bleu.compute(predictions=predictions, references=[[r] for r in references]))
print("chrF:", chrf.compute(predictions=predictions, references=[[r] for r in references]))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


                      Исходный текст                            Эталон                             Перевод
       The meeting starts on Monday. Встреча начинается в понедельник. Заседание начинается в понедельник.
The company did not increase prices.        Компания не повышала цены.          Компания не повысила цены.
BLEU: {'score': 45.180100180492246, 'counts': [8, 5, 2, 1], 'totals': [10, 8, 6, 4], 'precisions': [80.0, 62.5, 33.333333333333336, 25.0], 'bp': 1.0, 'sys_len': 10, 'ref_len': 10}
chrF: {'score': 76.29920935617433, 'char_order': 6, 'word_order': 0, 'beta': 2}


## 4. Диагностика ошибок перевода

Проверьте отдельно: числа и проценты, отрицание, имена собственные, терминологию, перевод длинных конструкций. При работе с NLLB рекомендуется использовать относительно короткие тексты; модель не оптимизирована для перевода длинных документов как единого блока.

In [5]:
diagnostic = [
    "The shipment includes 1,250 units, not 12,500 units.",
    "The regulator has not lifted the export restrictions.",
    "The contract was signed by Orion Technologies on 14 May 2025.",
]
for s in diagnostic:
    print("EN:", s, "\nRU:", translate(s), "\n")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: The shipment includes 1,250 units, not 12,500 units. 
RU: Перевозка включает в себя 1250 единиц, а не 12500 единиц. 



[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: The regulator has not lifted the export restrictions. 
RU: Регулятор не снял экспортные ограничения. 

EN: The contract was signed by Orion Technologies on 14 May 2025. 
RU: Контракт был подписан компанией Orion Technologies 14 мая 2025 года. 



## 5. Исследовательский эксперимент: NLLB для одноязычного реферирования

**Гипотеза:** параметры специализированной модели перевода можно адаптировать к другой задаче преобразования текста — генерации краткого содержания.

**Ограничения интерпретации:** модель NLLB дообучалась на переводе, а не на реферировании. Реферирование на английском языке при `src_lang=tgt_lang=eng_Latn` не является документированным штатным сценарием. Запуск может приводить к копированию входа, бессвязному тексту или отсутствию ожидаемой компрессии. Для строгого сравнения следует использовать одинаковые данные, бюджет обучения и несколько запусков, а также сопоставимую контрольную модель.

По умолчанию обучение **отключено**. Для его включения поменяйте `RUN_EXPERIMENTAL_FINETUNING` в блоке переменных на `True`. Запуск на CPU практически нецелесообразен.

In [7]:
RUN_EXPERIMENTAL_FINETUNING = True  # включать только при наличии подходящего GPU

In [10]:
if RUN_EXPERIMENTAL_FINETUNING:
    assert torch.cuda.is_available(), "Для данного эксперимента требуется GPU."
    # Убираем переводческое направление: и вход, и выход должны быть английскими.
    tokenizer.src_lang = "eng_Latn"
    tokenizer.tgt_lang = "eng_Latn"
    raw = load_dataset("duccd/billsum",
    data_files="ca_test.jsonl", split="train",).shuffle(seed=SEED)
    train_raw = raw.select(range(EXPERIMENT_TRAIN_SIZE))
    valid_raw = raw.select(range(EXPERIMENT_TRAIN_SIZE, EXPERIMENT_TRAIN_SIZE + EXPERIMENT_VALID_SIZE))

    def prepare(batch):
        inputs = tokenizer(batch["text"], max_length=MAX_SOURCE_LENGTH, truncation=True)
        targets = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
        inputs["labels"] = targets["input_ids"]
        return inputs

    train_ds = train_raw.map(prepare, batched=True, remove_columns=train_raw.column_names)
    valid_ds = valid_raw.map(prepare, batched=True, remove_columns=valid_raw.column_names)
    print("Размеры:", len(train_ds), len(valid_ds))
else:
    print("Дообучение отключено: теоретическую часть и перевод можно выполнить отдельно.")

Map: 100%|██████████| 16/16 [00:00<00:00, 634.59 examples/s]

Размеры: 64 16


### Дообучение через Seq2SeqTrainer

Для крупной модели размер пакета ограничен единицей. Даже при этом возможно исчерпание памяти. Если возникает ошибка нехватки памяти, не увеличивайте пакет; уменьшайте длины последовательностей, используйте подходящее ускорение или предварительно подготовленную контрольную точку.

При оценивании экспериментального реферирования принудительно задаём английский целевой язык.

In [12]:
if RUN_EXPERIMENTAL_FINETUNING:
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids("eng_Latn")
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
    args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR), max_steps=20, learning_rate=2e-5,
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=4, eval_strategy="no", save_strategy="no",
        logging_steps=5, report_to="none", fp16=True, seed=SEED,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                             eval_dataset=valid_ds, data_collator=collator, processing_class=tokenizer)
    result = trainer.train()
    print(result.metrics)
else:
    print("Пропуск обучения NLLB.")

Step,Training Loss
5,13.148254
10,12.229929
15,12.232139
20,11.863112


{'train_runtime': 391.8813, 'train_samples_per_second': 0.204, 'train_steps_per_second': 0.051, 'total_flos': 32506569031680.0, 'train_loss': 12.368358421325684, 'epoch': 1.25}


### Результаты и контрольная модель

Следующая ячейка показывает **только качественный пример** после экспериментального обучения. Для полноценного сравнения требуется получить ответы исходной NLLB **до** обучения на тех же документах, вычислить ROUGE на отдельной тестовой части и сопоставить с T5. Без этого нельзя делать вывод, какая модель лучше.

In [13]:
if RUN_EXPERIMENTAL_FINETUNING:
    tokenizer.src_lang = "eng_Latn"
    text = valid_raw[0]["text"]
    prepared = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=MAX_SOURCE_LENGTH).to(DEVICE)
    model.eval()
    with torch.inference_mode():
        result = model.generate(**prepared, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"))
    print("СГЕНЕРИРОВАННОЕ СОДЕРЖАНИЕ:", tokenizer.decode(result[0], skip_special_tokens=True))
    print("ЭТАЛОН:", valid_raw[0]["summary"][:700])
else:
    print("Образец будет получен после включения дообучения.")

[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


СГЕНЕРИРОВАННОЕ СОДЕРЖАНИЕ: The people of the State of California enact as follows: SECTION 1. Article 1 (starting with Section 18701) is added to Chapter 3 of Part 10.2 of Division 2 of the Revenue and Taxation Code, to read: Article 1. State Children's Trust Fund 18701. (a) An
ЭТАЛОН: Existing law allows individual taxpayers to contribute amounts in excess of their personal tax liability for the support of specified funds or accounts and previously allowed contributions to the State Children’s Trust Fund, which provides funding for child abuse and neglect prevention and intervention programs. This bill, for taxable years beginning on or after January 1, 2015, would allow individual taxpayers to contribute amounts in excess of their tax liability to the State Children’s Trust Fund. The bill would prohibit a voluntary contribution designation for this fund from being added on the form of the tax return until another designation is removed or space is available, whichever oc


## 6. Что можно поделать самостоятельно

1. Сравните прямой и обратный перевод; найдите смысловые изменения, не сводимые к выбору слов.
2. Проверьте перевод терминов и числовых данных на пяти специально подобранных предложениях.
3. Сформулируйте корректный протокол сравнения T5 и NLLB для реферирования: данные, бюджет, несколько запусков, метрики и ручная диагностика.
4. **При наличии GPU:** проведите экспериментальное дообучение и проверьте, выучила ли NLLB компрессию текста или лишь копирование.

### Источники
- [NLLB-200, карточка модели и ограничения](https://huggingface.co/facebook/nllb-200-distilled-600M)
- [NLLB в Transformers](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/nllb)
- [No Language Left Behind](https://arxiv.org/abs/2207.04672)
- [Seq2SeqTrainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)